# **1 CONFIG**

## 1.1 Google Sheets

In [ ]:
sheet_link = 'https://docs.google.com/spreadsheets/d/1aEx1tu29ZOBvS0bzi37b59l0nklKPHiLknss4EwbTYM/edit#gid=315917780'

sheet_name = 'PR- SPO'
sheet_range = 'A3:AB'

result_sheet_name = 'Result'
result_normal_PR = "Result final"
shortage_sheet_name = 'List P chưa có/thiếu PRs'

## 1.2 Parameters

In [ ]:
exchange_rate = 24000 # tỉ giá USD/VND
threshold_slobvalue = 100 # giới hạn slob value
threshold_leadtime = 16 # giới hạn lead time

# 2 IMPORT PACKAGE

In [ ]:
pip install pulp

In [ ]:
import pulp
import pandas as pd
import numpy as np
import datetime

import gspread
import gspread_dataframe as gd

from google.colab import data_table, auth
data_table.enable_dataframe_formatter()
auth.authenticate_user()

from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 3 READ AND CLEAN DATA

## 3.1 Read

In [ ]:
df_raw = pd.DataFrame.from_records(gc.open_by_url(sheet_link).worksheet(sheet_name).get(sheet_range))
df_raw.columns = df_raw.iloc[0,:]
df_raw = df_raw.iloc[1:,:]
df_raw.columns = df_raw.columns.str.strip()
df_raw

,Bundle,Status,SKU,Status SKU,Material,,Short Text,PGr,Quantity,Release Dt,...,Higlight SLOB,Shortage firm,Shortage N+1,FC - stock - PO pending,Qty safety,Standard price,Leadtime,Category,Demand firmed,PO
1,TW_16WAVY,C3,61029951,Discontinue,P11280025,,TW 16WAVY FHS ZIPPER BAG 5 1PK+C ASIA,27102621,"65,000",12/12/2023,...,,1823,-6904,-31992,0,870,16,#N/A,0,None
2,,C3,#N/A,Discontinue,P11280021,,EC 19RPP FHM ZIPPER BAG 4 1PK ASIA,27072701,"80,000",12/26/2023,...,,,,-33191,0,726,16,#N/A,0,None
3,,C8,#N/A,#N/A,P11270192,,CC 21 FHM FLOW WRAP 1PK+C 72 ASIA,27130623,"309,600",12/12/2023,...,#N/A,,,#N/A,0,250,16,#N/A,0,None
4,CC_21,C3,1050449,Active,P11270097,,Paperheader with string-151x102mm-PHI,27127451,"100,000",12/28/2023,...,,34867,7120,-72820,3000,657,7,Paper header,"62,763",None
5,,C3,#N/A,Active,P11260181,,CC 21 FHS INNER 3PK 72 US,27027368,"35,200",12/30/2023,...,,,,-42346,1000,3250,8,Inner,"40,360",None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
509,360RE_23,C3,61017963,Active,P11242183,,360RE 23 FHM SHIPPER 5PK INNER 60 US,27129678,"1,940",01/12/2024,...,,-607,-607,-6178,30,3798,7,Shipper,"7,407",None
510,360RE_23,C3,61017963,Active,P11242183,,360RE 23 FHM SHIPPER 5PK INNER 60 US,27129679,"1,600",01/19/2024,...,,-607,-607,-6178,30,3798,7,Shipper,"7,407",None
511,360RE_23,C3,61017963,Active,P11242183,,360RE 23 FHM SHIPPER 5PK INNER 60 US,27129680,"1,700",01/25/2024,...,,-607,-607,-6178,30,3798,7,Shipper,"7,407",None
512,360RE_23,C3,61017963,Active,P11212485,,360RE 23 FHM BC 5PK 120MM US,27129650,"20,400",01/18/2024,...,,17444,17444,-76540,5000,944,8,Backer card,"92,352",None


## 3.2 Clean

In [ ]:
df_raw = df_raw.astype(str)
df_raw = df_raw.applymap(lambda x: x.strip() if isinstance(x, str) else x)
df_raw = df_raw.replace({'#N/A': None, '#VALUE!': None, '#REF!': None, 'None': None, '': None})

df_raw['Order last lot'] = df_raw['Order last lot'].replace('Chưa order', None)
df_raw.loc[df_raw['Req.Date'] == 'Order 1 MOQ', 'Req.Date'] = df_raw['MOQ']

In [ ]:
df_raw['Release Dt'] = pd.to_datetime(df_raw['Release Dt'], format='%m/%d/%Y', errors='coerce')
df_raw = df_raw.sort_values(by=['Material', 'Release Dt'], ascending=[True, True])
df_raw['Release Dt'] = df_raw['Release Dt'].dt.strftime('%m/%d/%Y')

In [ ]:
numeric_columns = ['Quantity', 'MOQ', 'Req.Date', 'Order last lot', 'Shortage firm', 'Shortage N+1', 'FC - stock - PO pending', 'Qty safety', 'Standard price', 'Leadtime','Demand firmed']

# Replace characters in the specified columns
df_raw[numeric_columns] = df_raw[numeric_columns].replace({',': '', '\$': '', r'\(': '', r'\)': ''}, regex=True)
df_raw[numeric_columns] = df_raw[numeric_columns].apply(pd.to_numeric, errors='coerce')

In [ ]:
#df_raw[numeric_columns]

In [ ]:
df_raw.head()

,Bundle,Status,SKU,Status SKU,Material,,Short Text,PGr,Quantity,Release Dt,...,Higlight SLOB,Shortage firm,Shortage N+1,FC - stock - PO pending,Qty safety,Standard price,Leadtime,Category,Demand firmed,PO
497,K5+_TANDY,C3,61000478,Active,P11211766,None,KID 5+TANDY YHX BCARD 2PK 24 LATAM,27129374,152000,12/27/2023,...,None,64852.0,40176.0,-136836.0,5000,231,8,Backer card,200148,None
496,None,C3,None,Active,P11212007,None,ZZGC FHS BCARD 1PK T2x6(42) 72 RUS,27121664,50400,12/30/2023,...,None,NaN,NaN,-118249.0,5000,243,8,Backer card,163620,None
495,TW_16WAVY,C3,1051927,To be Discontinued,P11212008,None,TW FHS BCARD 5PK+3C T1x6(120) 120 PH,27123425,16800,12/28/2023,...,None,5570.0,1077.0,-978.0,0,530,8,Backer card,5990,None
494,EC_22RPP,C3,61023721,Active,P11212031,None,EC 19 rPP FHM BCARD 2PK T1X6 24 EUR,27129465,272000,12/12/2023,...,None,-26934.0,-118697.0,-114790.0,5000,240,8,Backer card,510996,None
493,None,C3,None,Active,P11212069,None,SF 13 FHS BCARD 2PK+C 24 LATAM,27116175,212000,12/30/2023,...,None,NaN,NaN,-430917.0,5000,224,8,Backer card,235791,None


## 3.3 Create needed columns

In [ ]:
df_temp = df_raw

In [ ]:
df_temp['Greater Option'] = np.nan
df_temp['Smaller Option'] = np.nan
df_temp['Greater Sum'] = np.nan
df_temp['Smaller Sum'] = np.nan

In [ ]:
df_temp['Classification'] = 'Unknown'
df_temp['SLOB'] = pd.NA
df_temp['SLOB Value'] = np.nan
df_temp['Choose'] = pd.NA
df_temp['Final Option'] = np.nan

In [ ]:
df_temp.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 513 entries, 497 to 1
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Bundle                   333 non-null    object 
 1   Status                   513 non-null    object 
 2   SKU                      349 non-null    object 
 3   Status SKU               502 non-null    object 
 4   Material                 513 non-null    object 
 5                            0 non-null      object 
 6   Short Text               513 non-null    object 
 7   PGr                      513 non-null    object 
 8   Quantity                 513 non-null    int64  
 9   Release Dt               513 non-null    object 
 10  Name of Vendor           512 non-null    object 
 11  MOQ                      513 non-null    int64  
 12  Req.Date                 349 non-null    float64
 13  Order last lot           24 non-null     float64
 14  Higlight SLOB            9

In [ ]:
df_temp.head()

,Bundle,Status,SKU,Status SKU,Material,,Short Text,PGr,Quantity,Release Dt,...,PO,Greater Option,Smaller Option,Greater Sum,Smaller Sum,Classification,SLOB,SLOB Value,Choose,Final Option
497,K5+_TANDY,C3,61000478,Active,P11211766,None,KID 5+TANDY YHX BCARD 2PK 24 LATAM,27129374,152000,12/27/2023,...,None,NaN,NaN,NaN,NaN,Unknown,<NA>,NaN,<NA>,NaN
496,None,C3,None,Active,P11212007,None,ZZGC FHS BCARD 1PK T2x6(42) 72 RUS,27121664,50400,12/30/2023,...,None,NaN,NaN,NaN,NaN,Unknown,<NA>,NaN,<NA>,NaN
495,TW_16WAVY,C3,1051927,To be Discontinued,P11212008,None,TW FHS BCARD 5PK+3C T1x6(120) 120 PH,27123425,16800,12/28/2023,...,None,NaN,NaN,NaN,NaN,Unknown,<NA>,NaN,<NA>,NaN
494,EC_22RPP,C3,61023721,Active,P11212031,None,EC 19 rPP FHM BCARD 2PK T1X6 24 EUR,27129465,272000,12/12/2023,...,None,NaN,NaN,NaN,NaN,Unknown,<NA>,NaN,<NA>,NaN
493,None,C3,None,Active,P11212069,None,SF 13 FHS BCARD 2PK+C 24 LATAM,27116175,212000,12/30/2023,...,None,NaN,NaN,NaN,NaN,Unknown,<NA>,NaN,<NA>,NaN


# 4 FUNCTION CHOOSE PR

Please check this video to understant the algorithm
[PuLP Tutorial: Linear Programming in Python](https://www.youtube.com/watch?v=qa4trkLfvwQ)

In [ ]:
def find_closest_pr_combinations(pr_quantity, requirement):
    prob_greater = pulp.LpProblem("ClosestToRequirementGreater", pulp.LpMinimize)
    prob_smaller = pulp.LpProblem("ClosestToRequirementSmaller", pulp.LpMinimize)

    # Define binary variables for each PR
    pr_vars_greater = [pulp.LpVariable(f'PR_{i}', 0, 1, pulp.LpInteger) for i in range(len(pr_quantity))]
    pr_vars_smaller = [pulp.LpVariable(f'PR_{i}', 0, 1, pulp.LpInteger) for i in range(len(pr_quantity))]

    # Define the objective functions for greater and smaller
    prob_greater += pulp.lpSum([pr_vars_greater[i] * pr_quantity[i] for i in range(len(pr_quantity))]) - requirement
    prob_smaller += requirement - pulp.lpSum([pr_vars_smaller[i] * pr_quantity[i] for i in range(len(pr_quantity))])

    # Add constraints to ensure that the total quantity is either greater or smaller than the requirement
    prob_greater += pulp.lpSum([pr_vars_greater[i] * pr_quantity[i] for i in range(len(pr_quantity))]) >= requirement
    prob_smaller += pulp.lpSum([pr_vars_smaller[i] * pr_quantity[i] for i in range(len(pr_quantity))]) <= requirement

    # Solve for greater and smaller scenarios
    prob_greater.solve()
    prob_smaller.solve()

    # Extract binary variables for both scenarios
    pr_binary_greater = [int(pr_vars_greater[i].value()) for i in range(len(pr_quantity))]
    pr_binary_smaller = [int(pr_vars_smaller[i].value()) for i in range(len(pr_quantity))]

    # Calculate the sums of PR quantities in both scenarios
    sum_pr_greater = sum(pr_quantity[i] for i in range(len(pr_quantity)) if pr_binary_greater[i] == 1)
    sum_pr_smaller = sum(pr_quantity[i] for i in range(len(pr_quantity)) if pr_binary_smaller[i] == 1)

    return pr_binary_greater, pr_binary_smaller, sum_pr_greater, sum_pr_smaller

# 5 CLASSIFICATION AND CHOOSE

## 5.1 Classification Material Change

In [ ]:
for index, row in df_temp.iterrows():
    if (pd.isnull(row['Req.Date'])) or (row['Req.Date'] == ''):
        df_temp.at[index, 'Classification'] = "Material Change"
        df_temp.at[index, 'Choose'] = "Material Change"

In [ ]:
df_temp_m_change = df_temp[df_temp['Classification'] == "Material Change"]
df_temp_m_change

,Bundle,Status,SKU,Status SKU,Material,,Short Text,PGr,Quantity,Release Dt,...,PO,Greater Option,Smaller Option,Greater Sum,Smaller Sum,Classification,SLOB,SLOB Value,Choose,Final Option
496,None,C3,None,Active,P11212007,None,ZZGC FHS BCARD 1PK T2x6(42) 72 RUS,27121664,50400,12/30/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
493,None,C3,None,Active,P11212069,None,SF 13 FHS BCARD 2PK+C 24 LATAM,27116175,212000,12/30/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
490,None,C3,None,Active,P11212114,None,EC 19 RPP FHM BCARD 1PK T2X6 72 AEA,27116177,50400,12/30/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
483,None,C3,None,Active,P11212211,None,TWGC 16 FHS BCARD 1PK T2X6 72 VN,27126166,50400,12/23/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
481,None,C3,None,Active,P11212229,None,PC REVITE FHM BCARD HG12 144 ASIA,27115447,97200,01/05/2024,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,None,C3,None,Discontinue,P11250950,None,STICKER TW 23 50X76 ROLL LATAM,27129351,30000,12/23/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
10,None,C3,None,Active,P11260175,None,PC 21 FHM 1PK-32MM INNER 14 84 LATAM,27130582,100000,01/02/2024,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
5,None,C3,None,Active,P11260181,None,CC 21 FHS INNER 3PK 72 US,27027368,35200,12/30/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
3,None,C8,None,None,P11270192,None,CC 21 FHM FLOW WRAP 1PK+C 72 ASIA,27130623,309600,12/12/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN


## 5.2 Apply function Choose PR

In [ ]:
df = pd.DataFrame()
for material, group in df_temp[~df_temp['Classification'].isin(["Material Change"])].groupby('Material'):
    try:
        pr_quantity = group['Quantity'].tolist()
        requirement = group['Req.Date'].max()  # Choose the maximum Req.Date as the requirement

        pr_binary_greater, pr_binary_smaller, sum_pr_greater, sum_pr_smaller = find_closest_pr_combinations(pr_quantity, requirement)

        group['Greater Option'] = pr_binary_greater
        group['Smaller Option'] = pr_binary_smaller
        group['Greater Sum'] = sum_pr_greater
        group['Smaller Sum'] = sum_pr_smaller

        df = pd.concat([df, group])

    except Exception as e:
        print(f"Error processing group for material {material}")
        continue

In [ ]:
df = pd.concat([df_temp_m_change, df], ignore_index=True)
df.reset_index(drop=True, inplace=True)
df

,Bundle,Status,SKU,Status SKU,Material,,Short Text,PGr,Quantity,Release Dt,...,PO,Greater Option,Smaller Option,Greater Sum,Smaller Sum,Classification,SLOB,SLOB Value,Choose,Final Option
0,None,C3,None,Active,P11212007,None,ZZGC FHS BCARD 1PK T2x6(42) 72 RUS,27121664,50400,12/30/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
1,None,C3,None,Active,P11212069,None,SF 13 FHS BCARD 2PK+C 24 LATAM,27116175,212000,12/30/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
2,None,C3,None,Active,P11212114,None,EC 19 RPP FHM BCARD 1PK T2X6 72 AEA,27116177,50400,12/30/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
3,None,C3,None,Active,P11212211,None,TWGC 16 FHS BCARD 1PK T2X6 72 VN,27126166,50400,12/23/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
4,None,C3,None,Active,P11212229,None,PC REVITE FHM BCARD HG12 144 ASIA,27115447,97200,01/05/2024,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
508,CC_21,C3,1050449,Active,P11270097,None,Paperheader with string-151x102mm-PHI,27127451,100000,12/28/2023,...,None,1.0,0.0,100000.0,0.0,Unknown,<NA>,NaN,NaN,NaN
509,CC_21,C3,61041164,Active,P11270196,None,CC 21 FHM FLOW WRAP 1PK PH,27119640,309600,12/30/2023,...,None,1.0,0.0,309600.0,0.0,Unknown,<NA>,NaN,NaN,NaN
510,CC_21,C3,61041164,Active,P11270196,None,CC 21 FHM FLOW WRAP 1PK PH,27112080,412800,01/11/2024,...,None,0.0,0.0,309600.0,0.0,Unknown,<NA>,NaN,NaN,NaN
511,CC_21,C3,61041164,Active,P11270196,None,CC 21 FHM FLOW WRAP 1PK PH,27120483,309600,01/20/2024,...,None,0.0,0.0,309600.0,0.0,Unknown,<NA>,NaN,NaN,NaN


## 5.3 Classification Last Lot

In [ ]:
for index, row in df[~df['Classification'].isin(["Material Change"])].iterrows():
    if (not pd.isnull(row['Order last lot'])) and (row['Order last lot'] != ''):
        df.at[index, 'Classification'] = "Last Lot"
        df.at[index, 'Choose'] = "Last Lot"

In [ ]:
# For PRs of Last Lot, assign 1 to the first PR and 0 to the rest
df['Index PR'] = df.groupby('Material').cumcount() + 1

df.loc[(df['Classification'] == 'Last Lot') & (df['Index PR'] == 1), 'Final Option'] = 1
df.loc[(df['Classification'] == 'Last Lot') & (df['Index PR'] != 1), 'Final Option'] = 0

df = df.drop('Index PR', axis=1)

## 5.4 Classification Safety PR

In [ ]:
for index, row in df[~df['Classification'].isin(["Material Change", "Last Lot"])].iterrows():
    if (row['Req.Date'] == row['Qty safety']) and (row['Demand firmed'] == 0):
        df.at[index, 'Classification'] = "Safety PR"
        df.at[index, 'Choose'] = "Safety PR"
        df.at[index, 'Final Option'] = 0 # hay "0"?!

## 5.5 Classification Normal PR & SLOB

In [ ]:
def result(Classification=None, SLOB=None, SLOB_Value=None, Choose=None, Final_Option=None, index=None, row=None, df=None, exchange_rate=None):
    if (index is not None) and (row is not None) and (df is not None) and (exchange_rate is not None):
        if Classification is not None:
            df.at[index, 'Classification'] = Classification

        if SLOB is not None and SLOB_Value is not None:
            df.at[index, 'SLOB'] = SLOB
            df.at[index, 'SLOB Value'] = (row[SLOB_Value] + row['FC - stock - PO pending']) * row['Standard price'] / exchange_rate

        if Choose is not None:
            df.at[index, 'Choose'] = Choose

        if Final_Option is not None:
            df.at[index, 'Final Option'] = row[Final_Option]

In [ ]:
for index, row in df[~df['Classification'].isin(["Material Change", "Last Lot", "Safety PR"])].iterrows():
    if row['Greater Sum'] + row['FC - stock - PO pending'] >= 0:
        if row['Smaller Sum'] + row['Shortage N+1'] >= 0:
            result(Classification="Normal PR", Choose="Smaller", Final_Option="Smaller Option", index=index, row=row, df=df, exchange_rate=exchange_rate)
        elif row['Smaller Sum'] + row['Shortage N+1'] < 0:
            if row['Leadtime'] >= threshold_leadtime:
                result(Classification="SLOB", SLOB="Acceptable SLOB", SLOB_Value="Greater Sum", Choose="Greater", Final_Option="Greater Option", index=index, row=row, df=df, exchange_rate=exchange_rate)
            elif row['Leadtime'] < threshold_leadtime:
                if row['Smaller Sum'] + row['Shortage firm'] >= 0:
                    if (row['Greater Sum'] + row['FC - stock - PO pending'])*row['Standard price']/exchange_rate >= threshold_slobvalue:
                        result(Classification="Normal PR", Choose="Smaller", Final_Option="Smaller Option", index=index, row=row, df=df, exchange_rate=exchange_rate)
                    elif (row['Greater Sum'] + row['FC - stock - PO pending'])*row['Standard price']/exchange_rate < threshold_slobvalue:
                        result(Classification="SLOB", SLOB="Acceptable SLOB", SLOB_Value="Greater Sum", Choose="Greater", Final_Option="Greater Option", index=index, row=row, df=df, exchange_rate=exchange_rate)
                elif row['Smaller Sum'] + row['Shortage firm'] < 0:
                    result(Classification="SLOB", SLOB="Acceptable SLOB", SLOB_Value="Greater Sum", Choose="Greater", Final_Option="Greater Option", index=index, row=row, df=df, exchange_rate=exchange_rate)
    elif row['Greater Sum'] + row['FC - stock - PO pending'] < 0:
        result(Classification="Normal PR", Choose="Greater", Final_Option="Greater Option", index=index, row=row, df=df, exchange_rate=exchange_rate)

In [ ]:
df

,Bundle,Status,SKU,Status SKU,Material,,Short Text,PGr,Quantity,Release Dt,...,PO,Greater Option,Smaller Option,Greater Sum,Smaller Sum,Classification,SLOB,SLOB Value,Choose,Final Option
0,None,C3,None,Active,P11212007,None,ZZGC FHS BCARD 1PK T2x6(42) 72 RUS,27121664,50400,12/30/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
1,None,C3,None,Active,P11212069,None,SF 13 FHS BCARD 2PK+C 24 LATAM,27116175,212000,12/30/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
2,None,C3,None,Active,P11212114,None,EC 19 RPP FHM BCARD 1PK T2X6 72 AEA,27116177,50400,12/30/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
3,None,C3,None,Active,P11212211,None,TWGC 16 FHS BCARD 1PK T2X6 72 VN,27126166,50400,12/23/2023,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
4,None,C3,None,Active,P11212229,None,PC REVITE FHM BCARD HG12 144 ASIA,27115447,97200,01/05/2024,...,None,NaN,NaN,NaN,NaN,Material Change,<NA>,NaN,Material Change,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
508,CC_21,C3,1050449,Active,P11270097,None,Paperheader with string-151x102mm-PHI,27127451,100000,12/28/2023,...,None,1.0,0.0,100000.0,0.0,Normal PR,<NA>,NaN,Smaller,0.0
509,CC_21,C3,61041164,Active,P11270196,None,CC 21 FHM FLOW WRAP 1PK PH,27119640,309600,12/30/2023,...,None,1.0,0.0,309600.0,0.0,Normal PR,<NA>,NaN,Greater,1.0
510,CC_21,C3,61041164,Active,P11270196,None,CC 21 FHM FLOW WRAP 1PK PH,27112080,412800,01/11/2024,...,None,0.0,0.0,309600.0,0.0,Normal PR,<NA>,NaN,Greater,0.0
511,CC_21,C3,61041164,Active,P11270196,None,CC 21 FHM FLOW WRAP 1PK PH,27120483,309600,01/20/2024,...,None,0.0,0.0,309600.0,0.0,Normal PR,<NA>,NaN,Greater,0.0


In [ ]:
# Assuming df is your original DataFrame
# Create a copy of df to avoid modifying the original DataFrame
df_final = df.copy()

# Apply filters to df_final
filter_condition = (
    (df_final['Classification'].isin(['Normal PR', 'SLOB', 'Last Lot'])) &
    (df_final['Final Option'] == 1)
)

df_final = df_final[filter_condition]

# Now df_final contains only the rows that satisfy the specified conditions


In [ ]:
df_final

,Bundle,Status,SKU,Status SKU,Material,,Short Text,PGr,Quantity,Release Dt,...,PO,Greater Option,Smaller Option,Greater Sum,Smaller Sum,Classification,SLOB,SLOB Value,Choose,Final Option
165,TW_16WAVY,C3,1051927,To be Discontinued,P11212008,None,TW FHS BCARD 5PK+3C T1x6(120) 120 PH,27123425,16800,12/28/2023,...,None,1.0,0.0,16800.0,0.0,Last Lot,<NA>,NaN,Last Lot,1.0
166,EC_22RPP,C3,61023721,Active,P11212031,None,EC 19 rPP FHM BCARD 2PK T1X6 24 EUR,27129465,272000,12/12/2023,...,None,1.0,0.0,272000.0,0.0,SLOB,Acceptable SLOB,1572.10,Greater,1.0
167,TWGC_23,C3,VN00592A,To be Discontinued,P11212081,None,TW GC 16 FHS BCARD 2PK 24 LATAM,27129629,32000,12/23/2023,...,None,1.0,0.0,32000.0,0.0,Normal PR,<NA>,NaN,Greater,1.0
169,EC_22RPP,C3,1052159,Active,P11212137,None,EC 19 FHM BCARD 1PK+C T2X3 72 ASIA,27127479,247800,12/12/2023,...,None,1.0,1.0,298200.0,247800.0,Normal PR,<NA>,NaN,Smaller,1.0
172,EC_22RPP,C3,CN07168A,Active,P11212161,None,EC 19 FHS BCARD 1PK 72 NA,27116496,71400,12/23/2023,...,None,1.0,0.0,71400.0,0.0,Normal PR,<NA>,NaN,Greater,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
504,K5+_TANDY,C3,61010273,Active,P11260180,None,KID 5+ TANDY YHX INNER 1PK 48 LATAM,27129620,10000,12/14/2023,...,None,1.0,0.0,10000.0,0.0,Normal PR,<NA>,NaN,Greater,1.0
505,360RE_23,C3,61017963,Active,P11260183,None,360RE 23 FHM INNER 5PK 120MM US,27129670,3875,01/11/2024,...,None,1.0,1.0,7125.0,10500.0,Normal PR,<NA>,NaN,Greater,1.0
506,360RE_23,C3,61017963,Active,P11260183,None,360RE 23 FHM INNER 5PK 120MM US,27129671,3250,01/18/2024,...,None,1.0,1.0,7125.0,10500.0,Normal PR,<NA>,NaN,Greater,1.0
509,CC_21,C3,61041164,Active,P11270196,None,CC 21 FHM FLOW WRAP 1PK PH,27119640,309600,12/30/2023,...,None,1.0,0.0,309600.0,0.0,Normal PR,<NA>,NaN,Greater,1.0


# 6 WRITE RESULT

In [ ]:
# Print all result
result_link = sheet_link
result_wb = gc.open_by_url(result_link)
# result_wb.worksheet(result_sheet_name).clear()

result_sheet = result_wb.worksheet(result_sheet_name)

# Define the range to clear (from A1 to V{num_rows})
num_rows = result_sheet.row_count
range_to_clear = f'A1:AG{num_rows}'

# Clear the specific range
result_sheet.update(range_to_clear, [[]])


gd.set_with_dataframe(result_wb.worksheet(result_sheet_name),df)

In [ ]:
# Print choosen PRs only
result_sheet_final = result_wb.worksheet(result_normal_PR)

# Define the range to clear (from A1 to V{num_rows})
num_rows = result_sheet_final.row_count
range_to_clear = f'A1:AG{num_rows}'

# Clear the specific range
result_sheet_final.update(range_to_clear, [[]])

gd.set_with_dataframe(result_wb.worksheet(result_normal_PR),df_final)

# 7 RETURN LIST P THAT SHORTAGE PR QUANTITY

In [ ]:
# Applying conditions to create df_shortage
df_shortage = df[(~df['Classification'].isin(["Material Change", "Last Lot", "Safety PR"])) & (df['Final Option'] != 0) & (df['Final Option'] != 1)][['Material']]
df_shortage = df_shortage.drop_duplicates()
df_shortage['Material'].unique()

In [ ]:
shortage_link = sheet_link
shortage_wb = gc.open_by_url(shortage_link)

# Clear the range C2:C
shortage_wb.values_clear(f"{shortage_sheet_name}!C2:C")

# Set the new data in column C starting from row 2
data_to_set = df_shortage[['Material']].values.tolist()
range_to_set = f'C2:C{len(data_to_set)+1}'
shortage_wb.worksheet(shortage_sheet_name).update(range_to_set, data_to_set)

{'spreadsheetId': '1aEx1tu29ZOBvS0bzi37b59l0nklKPHiLknss4EwbTYM',
 'updatedRange': "'List P chưa có/thiếu PRs'!C2:C5",
 'updatedRows': 4,
 'updatedColumns': 1,
 'updatedCells': 4}

In [ ]:
print('Done')

Done


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 513 entries, 0 to 512
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Bundle                   333 non-null    object 
 1   Status                   513 non-null    object 
 2   SKU                      349 non-null    object 
 3   Status SKU               502 non-null    object 
 4   Material                 513 non-null    object 
 5                            0 non-null      object 
 6   Short Text               513 non-null    object 
 7   PGr                      513 non-null    object 
 8   Quantity                 513 non-null    int64  
 9   Release Dt               513 non-null    object 
 10  Name of Vendor           512 non-null    object 
 11  MOQ                      513 non-null    int64  
 12  Req.Date                 349 non-null    float64
 13  Order last lot           24 non-null     float64
 14  Higlight SLOB            9